In [ ]:
import os
import cv2

def main():
    folder = "../ImageWithCoins"

    print("Mostrando imagens...\n")

    for file in os.listdir(folder):

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            path = os.path.join(folder, file)

            img = cv2.imread(path)

            if img is None:
                print("Erro ao carregar:", file)
                continue

            print("Mostrando:", file)

            resized = cv2.resize(img, (800, 600))
            cv2.imshow("Imagem", resized)
            cv2.waitKey(0)  

    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

In [ ]:
import cv2
import numpy as np

img = cv2.imread("../ImageWithCoins/CoinsBlue1.jpeg")

hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

lower_blue = np.array([90, 50, 50])
upper_blue = np.array([140, 255, 255])

mask = cv2.inRange(hsv, lower_blue, upper_blue)

result = cv2.bitwise_and(img, img, mask=mask)

cv2.imshow("Original", img)
cv2.imshow("Mask", mask)
cv2.imshow("Resultado", result)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
import cv2
import numpy as np

img = cv2.imread("../ImageWithCoins/CoinsBlue1.jpeg")

img = cv2.resize(img, (800, 600))

hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

lower_blue = np.array([90, 80, 50])
upper_blue = np.array([140, 255, 255])

mask = cv2.inRange(hsv, lower_blue, upper_blue)

kernel = np.ones((5,5), np.uint8)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if contours:
    c = max(contours, key=cv2.contourArea)

    clean_mask = np.zeros_like(mask)
    cv2.drawContours(clean_mask, [c], -1, 255, -1)

    result = cv2.bitwise_and(img, img, mask=clean_mask)

    cv2.imshow("Original", img)
    cv2.imshow("Mask EVA", clean_mask)
    cv2.imshow("Resultado", result)

    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [ ]:
contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

c = max(contours, key=cv2.contourArea)

x, y, w, h = cv2.boundingRect(c)

cropped = img[y:y+h, x:x+w]

cv2.imshow("Recortado", cropped)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
import os
import cv2
import numpy as np

def process_folder(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for file in os.listdir(input_folder):

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            path = os.path.join(input_folder, file)
            img = cv2.imread(path)

            if img is None:
                print("Erro:", file)
                continue

            img = cv2.resize(img, (800, 600))

            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

            lower_blue = np.array([90, 80, 50])
            upper_blue = np.array([140, 255, 255])

            mask = cv2.inRange(hsv, lower_blue, upper_blue)

            kernel = np.ones((5,5), np.uint8)
            mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            if not contours:
                print("Sem contorno:", file)
                continue

            c = max(contours, key=cv2.contourArea)

            clean_mask = np.zeros_like(mask)
            cv2.drawContours(clean_mask, [c], -1, 255, -1)

            result = cv2.bitwise_and(img, img, mask=clean_mask)

            x, y, w, h = cv2.boundingRect(c)
            cropped = result[y:y+h, x:x+w]

            output_path = os.path.join(output_folder, file)
            cv2.imwrite(output_path, cropped)

            print("Salvo:", file)


def main():
    input_folder = "../ImageWithCoins"
    output_folder = "../ProcessedImages"

    process_folder(input_folder, output_folder)


if __name__ == "__main__":
    main()

In [ ]:
import os
import cv2
import numpy as np

def crop_all(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for file in os.listdir(input_folder):

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            path = os.path.join(input_folder, file)
            img = cv2.imread(path)

            if img is None:
                print("Erro:", file)
                continue

            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

            lower_blue = np.array([90, 80, 50])
            upper_blue = np.array([140, 255, 255])

            mask = cv2.inRange(hsv, lower_blue, upper_blue)

            kernel = np.ones((5,5), np.uint8)
            mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            if not contours:
                print("Sem contorno:", file)
                continue

            c = max(contours, key=cv2.contourArea)

            x, y, w, h = cv2.boundingRect(c)
            cropped = img[y:y+h, x:x+w]

            h2, w2 = cropped.shape[:2]
            scale = 600 / w2
            resized = cv2.resize(cropped, (int(w2*scale), int(h2*scale)))

            cv2.imshow("Recortado", resized)
            cv2.waitKey(0)
            cv2.destroyAllWindows()

            output_path = os.path.join(output_folder, file)
            cv2.imwrite(output_path, cropped)

            print("Recortado:", file)


def main():
    input_folder = "../ImageWithCoins"
    output_folder = "../CroppedImages"

    crop_all(input_folder, output_folder)


if __name__ == "__main__":
    main()

In [ ]:
import os
import cv2
import numpy as np

def count_coins(input_folder):

    print("\nContagem de moedas:\n")

    for file in os.listdir(input_folder):

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            path = os.path.join(input_folder, file)
            img = cv2.imread(path)

            if img is None:
                print("Erro:", file)
                continue

            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

            lower_blue = np.array([90, 80, 50])
            upper_blue = np.array([140, 255, 255])

            mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)

            mask_coins = cv2.bitwise_not(mask_blue)

            kernel = np.ones((5,5), np.uint8)
            mask_coins = cv2.morphologyEx(mask_coins, cv2.MORPH_OPEN, kernel)
            mask_coins = cv2.morphologyEx(mask_coins, cv2.MORPH_CLOSE, kernel)

            contours, _ = cv2.findContours(
                mask_coins, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )

            count = 0

            for c in contours:
                area = cv2.contourArea(c)

                if 2000 < area < 50000:
                    count += 1

            print(f"{file}: {count} moedas")

            show = cv2.resize(mask_coins, (600, 600))
            cv2.imshow("Mask Coins", show)
            cv2.waitKey(0)

    cv2.destroyAllWindows()


def main():
    input_folder = "../CroppedImages"  # 🔥 usa as imagens já limpas

    count_coins(input_folder)


if __name__ == "__main__":
    main()

In [ ]:
import os
import cv2
import numpy as np

def count_coins(input_folder):

    print("\nContagem de moedas:\n")

    for file in os.listdir(input_folder):

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            path = os.path.join(input_folder, file)
            img = cv2.imread(path)

            if img is None:
                print("Erro:", file)
                continue

            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

            lower_blue = np.array([90, 80, 50])
            upper_blue = np.array([140, 255, 255])

            mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)

            mask_coins = cv2.bitwise_not(mask_blue)

            kernel = np.ones((7,7), np.uint8)

            mask_coins = cv2.morphologyEx(mask_coins, cv2.MORPH_OPEN, kernel)

            mask_coins = cv2.morphologyEx(mask_coins, cv2.MORPH_CLOSE, kernel)

            mask_coins = cv2.erode(mask_coins, kernel, iterations=1)

            contours, _ = cv2.findContours(
                mask_coins, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )

            count = 0

            for c in contours:
                area = cv2.contourArea(c)

                if area < 3000 or area > 60000:
                    continue

                peri = cv2.arcLength(c, True)

                if peri == 0:
                    continue

                circularidade = 4 * np.pi * area / (peri * peri)

                if 0.75 < circularidade < 1.2:
                    count += 1

            print(f"{file}: {count} moedas")

            show = cv2.resize(mask_coins, (600, 600))
            cv2.imshow("Mask Coins", show)
            cv2.waitKey(0)

    cv2.destroyAllWindows()


def main():
    input_folder = "../CroppedImages"

    count_coins(input_folder)


if __name__ == "__main__":
    main()

In [ ]:
import os
import cv2
import numpy as np

def count_coins(input_folder):

    print("\nContagem de moedas:\n")

    for file in os.listdir(input_folder):

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            path = os.path.join(input_folder, file)
            img = cv2.imread(path)

            if img is None:
                print("Erro:", file)
                continue

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            gray = cv2.GaussianBlur(gray, (9, 9), 2)

            circles = cv2.HoughCircles(
                gray,
                cv2.HOUGH_GRADIENT,
                dp=1.2,
                minDist=50,
                param1=100,
                param2=30,
                minRadius=20,
                maxRadius=80
            )

            count = 0

            if circles is not None:
                circles = np.uint16(np.around(circles))
                count = len(circles[0])

                for (x, y, r) in circles[0]:
                    cv2.circle(img, (x, y), r, (0, 255, 0), 2)

            print(f"{file}: {count} moedas")

            show = cv2.resize(img, (600, 600))
            cv2.imshow("Deteccao", show)
            cv2.waitKey(0)

    cv2.destroyAllWindows()


def main():
    input_folder = "../CroppedImages"

    count_coins(input_folder)


if __name__ == "__main__":
    main()

In [ ]:
import os
import cv2
import numpy as np

def count_coins(input_folder):
    print("\nContagem de moedas:\n")

    for file in os.listdir(input_folder):
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            path = os.path.join(input_folder, file)
            img = cv2.imread(path)

            if img is None:
                print("Erro:", file)
                continue

            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

            lower_blue = np.array([90, 80, 50])
            upper_blue = np.array([140, 255, 255])
            mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)

            mask_pieces = cv2.bitwise_not(mask_blue)

            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
            mask_pieces = cv2.morphologyEx(mask_pieces, cv2.MORPH_OPEN, kernel)
            mask_pieces = cv2.morphologyEx(mask_pieces, cv2.MORPH_CLOSE, kernel)

            blur = cv2.GaussianBlur(mask_pieces, (9, 9), 2)

            circles = cv2.HoughCircles(
                blur,
                cv2.HOUGH_GRADIENT,
                dp=1.2,
                minDist=60,
                param1=100,
                param2=18,
                minRadius=25,
                maxRadius=55
            )

            count = 0

            if circles is not None:
                circles = np.uint16(np.around(circles))
                valid_circles = []

                for (x, y, r) in circles[0]:
                    if x - r < 5 or y - r < 5 or x + r > img.shape[1] - 5 or y + r > img.shape[0] - 5:
                        continue

                    valid_circles.append((x, y, r))

                count = len(valid_circles)

                for (x, y, r) in valid_circles:
                    cv2.circle(img, (x, y), r, (0, 255, 0), 2)
                    cv2.circle(img, (x, y), 2, (0, 0, 255), 3)

            print(f"{file}: {count} moedas")

            show_mask = cv2.resize(mask_pieces, (600, 600))
            show_img = cv2.resize(img, (600, 600))

            cv2.imshow("Mask Pieces", show_mask)
            cv2.imshow("Deteccao", show_img)
            cv2.waitKey(0)

    cv2.destroyAllWindows()

def main():
    input_folder = "../CroppedImages"
    count_coins(input_folder)

if __name__ == "__main__":
    main()

In [ ]:
import os
import cv2
import numpy as np

def count_coins(input_folder):

    real_values = {
        
        "CoinsBlue1.jpeg": 10,
    "CoinsBlue2.jpeg": 7,
    "CoinsBlue3.jpeg": 5,
    "CoinsBlue4.jpeg": 3,
    "CoinsBlue5.jpeg": 2,
    "CoinsBlue6.jpeg": 4,
    "CoinsBlue7.jpeg": 6,
    "CoinsBlue8.jpeg": 7,
    "CoinsBlue9.jpeg": 7,
    "CoinsBlue10.jpeg": 7,
    "CoinsBlue11.jpeg": 1,
    "CoinsBlue12.jpeg": 3,
    "CoinsBlue13.jpeg": 2,
    "CoinsBlue14.jpeg": 2,
    "CoinsBlue15.jpeg": 3,
    "CoinsBlue16.jpeg": 3,
    "CoinsBlue17.jpeg": 3,
    "CoinsBlue18.jpeg": 10,
    "CoinsBlue19.jpeg": 10,
    "CoinsBlue20.jpeg": 10,
    "CoinsBlue21.jpeg": 5,
    "CoinsBlue22.jpeg": 5,
    "CoinsBlue23.jpeg": 5,
        
    }

    resultados = []

    for file in os.listdir(input_folder):

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            path = os.path.join(input_folder, file)
            img = cv2.imread(path)

            if img is None:
                continue

            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

            lower_blue = np.array([90, 80, 50])
            upper_blue = np.array([140, 255, 255])

            mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)
            mask_pieces = cv2.bitwise_not(mask_blue)

            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
            mask_pieces = cv2.morphologyEx(mask_pieces, cv2.MORPH_OPEN, kernel)
            mask_pieces = cv2.morphologyEx(mask_pieces, cv2.MORPH_CLOSE, kernel)

            blur = cv2.GaussianBlur(mask_pieces, (9, 9), 2)

            circles = cv2.HoughCircles(
                blur,
                cv2.HOUGH_GRADIENT,
                dp=1.2,
                minDist=60,
                param1=100,
                param2=18,
                minRadius=25,
                maxRadius=55
            )

            detectado = 0

            if circles is not None:
                circles = np.uint16(np.around(circles))
                detectado = len(circles[0])

            real = real_values.get(file, 0)
            erro = abs(real - detectado)

            resultados.append((file, real, detectado, erro))

    # 🔥 salvar CSV
    with open("resultados.csv", "w") as f:
        f.write("Imagem,Real,Detectado,Erro\n")

        for r in resultados:
            f.write(f"{r[0]},{r[1]},{r[2]},{r[3]}\n")

    print("\nTabela salva como resultados.csv")


def main():
    input_folder = "../CroppedImages"
    count_coins(input_folder)


if __name__ == "__main__":
    main()


Tabela salva como resultados.csv


In [ ]:
import os
import cv2
import numpy as np

def count_coins(input_folder):

    real_values = {
        "CoinsBlue1.jpeg": 10,
        "CoinsBlue2.jpeg": 7,
        "CoinsBlue3.jpeg": 5,
        "CoinsBlue4.jpeg": 3,
        "CoinsBlue5.jpeg": 2,
        "CoinsBlue6.jpeg": 4,
        "CoinsBlue7.jpeg": 6,
        "CoinsBlue8.jpeg": 7,
        "CoinsBlue9.jpeg": 7,
        "CoinsBlue10.jpeg": 7,
        "CoinsBlue11.jpeg": 1,
        "CoinsBlue12.jpeg": 3,
        "CoinsBlue13.jpeg": 2,
        "CoinsBlue14.jpeg": 2,
        "CoinsBlue15.jpeg": 3,
        "CoinsBlue16.jpeg": 3,
        "CoinsBlue17.jpeg": 3,
        "CoinsBlue18.jpeg": 10,
        "CoinsBlue19.jpeg": 10,
        "CoinsBlue20.jpeg": 10,
        "CoinsBlue21.jpeg": 5,
        "CoinsBlue22.jpeg": 5,
        "CoinsBlue23.jpeg": 5,
        
        
    }

    resultados = []

    print("\nProcessando imagens...\n")

    for file in os.listdir(input_folder):

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            path = os.path.join(input_folder, file)
            img = cv2.imread(path)

            if img is None:
                print("Erro:", file)
                continue

            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

            lower_blue = np.array([90, 80, 50])
            upper_blue = np.array([140, 255, 255])

            mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)

            mask_pieces = cv2.bitwise_not(mask_blue)

            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
            mask_pieces = cv2.morphologyEx(mask_pieces, cv2.MORPH_OPEN, kernel)
            mask_pieces = cv2.morphologyEx(mask_pieces, cv2.MORPH_CLOSE, kernel)

            blur = cv2.GaussianBlur(mask_pieces, (9, 9), 2)

            circles = cv2.HoughCircles(
                blur,
                cv2.HOUGH_GRADIENT,
                dp=1.2,
                minDist=60,
                param1=100,
                param2=18,
                minRadius=25,
                maxRadius=55
            )

            detectado = 0

            if circles is not None:
                circles = np.uint16(np.around(circles))

                valid_circles = []

                for (x, y, r) in circles[0]:
                    if x - r < 5 or y - r < 5 or x + r > img.shape[1] - 5 or y + r > img.shape[0] - 5:
                        continue

                    valid_circles.append((x, y, r))

                detectado = len(valid_circles)

            real = real_values.get(file, 0)

            erro = abs(real - detectado)

            resultados.append((file, real, detectado, erro))

    with open("resultados.csv", "w") as f:
        f.write("Imagem,Real,Detectado,Erro\n")

        for r in resultados:
            f.write(f"{r[0]},{r[1]},{r[2]},{r[3]}\n")

    print("\nTabela salva como resultados.csv")

    print("\nRESULTADOS:\n")
    print(f"{'Imagem':20} {'Real':5} {'Detectado':10} {'Erro':5}")

    for r in resultados:
        print(f"{r[0]:20} {r[1]:5} {r[2]:10} {r[3]:5}")

    media = sum(r[3] for r in resultados) / len(resultados)
    print(f"\nErro médio: {media:.2f}")


def main():
    input_folder = "../CroppedImages"

    count_coins(input_folder)


if __name__ == "__main__":
    main()